In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import mlflow

from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight

# --- 1. CONFIGURACIÓN DE MLFLOW ---  PEDIR CREDENCIALES!!!!!
os.environ["MLFLOW_TRACKING_URI"] = ""
os.environ["MLFLOW_TRACKING_USERNAME"] = ""
os.environ["MLFLOW_TRACKING_PASSWORD"] = ""

mlflow.set_experiment("XGBoost_BioVid_G1")
mlflow.xgboost.autolog() # ¡El espía automático de métricas!

# --- 2. CARGA Y PREPARACIÓN DE DATOS ---
df = pd.read_csv('/content/drive/Shareddrives/DATA_GPI/Grupo1_Desarrollo/Entregable Grupo Uno/Actividad 4/dataset_features_biovid_normalizado.csv')

print(df.head(0))

# Mapeo obligatorio para XGBoost (Target BioVid debe ser 0, 1, 2, 3, 4)
df['target'] = df['class_name'].map({'BL1': 0, 'PA1': 1, 'PA2': 2, 'PA3': 3, 'PA4': 4})

# Seleccionar Features (Ajustadas al df)
features = ['ecg_bpm_neto', 'gsr_peaks_count_neto', 'gsr_max_amplitude_neto', 'emg_trapezius_auc_neto', 'emg_corrugator_auc_neto', 'emg_zygomaticus_auc_neto']
X = df[features]
y = df['target']
grupos = df['subject_name'] # Clave para que GroupKFold aísle a los pacientes en BioVid

# --- 3. ENTRENAMIENTO AISLADO (GROUP K-FOLD) ---
gkf = GroupKFold(n_splits=5)

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=grupos)):
    with mlflow.start_run(run_name=f"XGBoost_BioVid_Fold_{fold+1}"):
        print(f"--- Iniciando Entrenamiento Fold {fold+1} ---")

        # Separar datos
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_test, y_test   = X.iloc[test_idx], y.iloc[test_idx]

        # Calcular pesos para mitigar el Desbalance de Clases
        pesos_train = compute_sample_weight('balanced', y_train)

        # Configurar y Entrenar Modelo (num_class es 5 porque hay 5 niveles de BL1 a PA4)
        modelo = xgb.XGBClassifier(objective='multi:softmax', num_class=5, max_depth=4, random_state=42)
        modelo.fit(X_train, y_train, sample_weight=pesos_train, eval_set=[(X_test, y_test)], verbose=False)

        # Evaluar
        preds = modelo.predict(X_test)
        macro_f1 = f1_score(y_test, preds, average='macro')

        # Guardar métrica clave en MLflow manualmente (autolog guarda el resto)
        mlflow.log_metric("val_macro_f1", macro_f1)
        print(f"Fold {fold+1} Finalizado - Macro F1-Score: {macro_f1:.4f}\n")

print("¡Experimentos guardados exitosamente en MLflow!")

Empty DataFrame
Columns: [subject_name, class_id, class_name, ecg_bpm_neto, gsr_peaks_count_neto, gsr_max_amplitude_neto, emg_trapezius_auc_neto, emg_corrugator_auc_neto, emg_zygomaticus_auc_neto]
Index: []
--- Iniciando Entrenamiento Fold 1 ---


2026/06/08 19:52:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Fold 1 Finalizado - Macro F1-Score: 0.2514

🏃 View run XGBoost_BioVid_Fold_1 at: https://dagshub.com/GaboGolCs/PainPredict-Neuro.mlflow/#/experiments/1/runs/731a044e0fb94df7baf53a1087f9ea16
🧪 View experiment at: https://dagshub.com/GaboGolCs/PainPredict-Neuro.mlflow/#/experiments/1
--- Iniciando Entrenamiento Fold 2 ---


2026/06/08 19:52:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Fold 2 Finalizado - Macro F1-Score: 0.3001

🏃 View run XGBoost_BioVid_Fold_2 at: https://dagshub.com/GaboGolCs/PainPredict-Neuro.mlflow/#/experiments/1/runs/483b65c5eabb47cd9582b36de941b3ab
🧪 View experiment at: https://dagshub.com/GaboGolCs/PainPredict-Neuro.mlflow/#/experiments/1
--- Iniciando Entrenamiento Fold 3 ---


...

Fold 3 Finalizado - Macro F1-Score: 0.2660

🏃 View run XGBoost_BioVid_Fold_3 at: https://dagshub.com/GaboGolCs/PainPredict-Neuro.mlflow/#/experiments/1/runs/b3857d727d584c3c9a87c09a5310c259
🧪 View experiment at: https://dagshub.com/GaboGolCs/PainPredict-Neuro.mlflow/#/experiments/1
--- Iniciando Entrenamiento Fold 4 ---


2026/06/08 19:54:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Fold 4 Finalizado - Macro F1-Score: 0.2738

🏃 View run XGBoost_BioVid_Fold_4 at: https://dagshub.com/GaboGolCs/PainPredict-Neuro.mlflow/#/experiments/1/runs/6a6fda27776f4903848c36eabdbb8e6d
🧪 View experiment at: https://dagshub.com/GaboGolCs/PainPredict-Neuro.mlflow/#/experiments/1
--- Iniciando Entrenamiento Fold 5 ---


2026/06/08 19:54:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Fold 5 Finalizado - Macro F1-Score: 0.2642

🏃 View run XGBoost_BioVid_Fold_5 at: https://dagshub.com/GaboGolCs/PainPredict-Neuro.mlflow/#/experiments/1/runs/a92423d922a54dcb9eaa864e8e5d9323
🧪 View experiment at: https://dagshub.com/GaboGolCs/PainPredict-Neuro.mlflow/#/experiments/1
¡Experimentos guardados exitosamente en MLflow!
